# 04 · Qualidade dos dados

Três níveis de verificação, todos gravados em tabelas da gold para consulta e auditoria:

1. **Atributos da origem** (`dq_bronze_atributos`): ausências e cardinalidade de cada uma das 52 colunas dos nove arquivos, antes de qualquer tratamento.
2. **Atributos do modelo** (`dq_gold_atributos`): para cada coluna do Catálogo de Dados, completude, obrigatoriedade, cardinalidade, mínimo, máximo e quantidade de valores fora do domínio declarado.
3. **Regras** (`dq_regras`): unicidade de chaves, integridade referencial, consistência temporal, acurácia geográfica e atualidade.

Valor fora do domínio não é necessariamente erro: o domínio do catálogo é a faixa *esperada*, e o que escapa dela precisa ser examinado e explicado.

In [ ]:
import os
import pandas as pd
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "workspace", "Catálogo do Unity Catalog")
CATALOGO = dbutils.widgets.get("catalogo")
BRONZE, SILVER, GOLD = (f"{CATALOGO}.olist_{c}" for c in ("bronze", "silver", "gold"))
catalogo = pd.read_csv(os.path.join(os.getcwd(), "..", "catalogo", "catalogo_dados.csv"), sep=";", dtype=str).fillna("")

def gravar(pdf, nome, descricao):
    spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{GOLD}.{nome}")
    spark.sql(f"COMMENT ON TABLE {GOLD}.{nome} IS '{descricao}'")

## 1. Atributos da origem
Na bronze tudo é texto; ausência é nulo ou texto vazio.

In [ ]:
linhas = []
for t in [r.tableName for r in spark.sql(f"SHOW TABLES IN {BRONZE}").collect() if r.tableName != "registro_coleta"]:
    df = spark.table(f"{BRONZE}.{t}")
    cols = [c for c in df.columns if not c.startswith("_")]
    vazio = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == "")
    r = df.agg(F.count("*").alias("_n"),
               *[F.sum(vazio(c).cast("int")).alias(f"nulo_{c}") for c in cols],
               *[F.countDistinct(c).alias(f"dist_{c}") for c in cols]).first()
    for c in cols:
        linhas.append({"tabela_origem": t, "atributo": c, "linhas": r["_n"], "ausentes": r[f"nulo_{c}"],
                       "pct_ausentes": round(100 * r[f"nulo_{c}"] / r["_n"], 2),
                       "distintos": r[f"dist_{c}"]})
dq_bronze = pd.DataFrame(linhas)
gravar(dq_bronze, "dq_bronze_atributos", "Qualidade: ausências e cardinalidade de cada coluna da origem")
display(dq_bronze.sort_values("pct_ausentes", ascending=False))

tabela_origem,atributo,linhas,ausentes,pct_ausentes,distintos
order_reviews,review_comment_title,99224,87658,88.34,4527
order_reviews,review_comment_message,99224,58256,58.71,36159
orders,order_delivered_customer_date,99441,2965,2.98,95664
products,product_description_lenght,32951,610,1.85,2960
products,product_photos_qty,32951,610,1.85,19
products,product_category_name,32951,610,1.85,73
products,product_name_lenght,32951,610,1.85,66
orders,order_delivered_carrier_date,99441,1783,1.79,81018
orders,order_approved_at,99441,160,0.16,90733
products,product_weight_g,32951,2,0.01,2204


## 2. Atributos do modelo contra o Catálogo de Dados

In [ ]:
def fora_do_dominio(col, tipo_spark, r):
    if r.categorias:
        return ~F.lower(col.cast("string")).isin([x.lower() for x in r.categorias.split("|")])
    cond = F.lit(False)
    if r.dominio_min:
        cond = cond | (col < F.lit(r.dominio_min).cast(tipo_spark))
    if r.dominio_max:
        cond = cond | (col > F.lit(r.dominio_max).cast(tipo_spark))
    return cond

linhas = []
for tabela, grupo in catalogo.groupby("tabela", sort=False):
    df = spark.table(f"{GOLD}.{tabela}")
    tipos = dict(df.dtypes)
    exprs = [F.count("*").alias("_n")]
    for r in grupo.itertuples():
        c = F.col(r.atributo)
        exprs += [F.sum(c.isNull().cast("int")).alias(f"nulo_{r.atributo}"),
                  F.countDistinct(c).alias(f"dist_{r.atributo}"),
                  F.min(c).cast("string").alias(f"min_{r.atributo}"),
                  F.max(c).cast("string").alias(f"max_{r.atributo}"),
                  F.sum((c.isNotNull() & fora_do_dominio(c, tipos[r.atributo], r)).cast("int")).alias(f"fora_{r.atributo}")]
    agg = df.agg(*exprs).first()
    for r in grupo.itertuples():
        nulos = agg[f"nulo_{r.atributo}"]
        linhas.append({"tabela": tabela, "atributo": r.atributo, "tipo": r.tipo, "linhas": agg["_n"],
                       "nulos": nulos, "pct_nulos": round(100 * nulos / agg["_n"], 2),
                       "obrigatorio": r.obrigatorio, "viola_obrigatoriedade": r.obrigatorio == "sim" and nulos > 0,
                       "distintos": agg[f"dist_{r.atributo}"],
                       "minimo": agg[f"min_{r.atributo}"], "maximo": agg[f"max_{r.atributo}"],
                       "dominio": r.categorias or (f"[{r.dominio_min}, {r.dominio_max}]" if r.dominio_min else "livre"),
                       "fora_do_dominio": agg[f"fora_{r.atributo}"]})
dq_gold = pd.DataFrame(linhas)
gravar(dq_gold, "dq_gold_atributos", "Qualidade: cada coluna do modelo conferida contra o Catálogo de Dados")
display(dq_gold)

tabela,atributo,tipo,linhas,nulos,pct_nulos,obrigatorio,viola_obrigatoriedade,distintos,minimo,maximo,dominio,fora_do_dominio
dim_data,data_key,numérico,1827,0,0.0,sim,False,1827,20160101,20201231,"[20160101, 20201231]",0
dim_data,data,data,1827,0,0.0,sim,False,1827,2016-01-01,2020-12-31,"[2016-01-01, 2020-12-31]",0
dim_data,ano,numérico,1827,0,0.0,sim,False,5,2016,2020,"[2016, 2020]",0
dim_data,trimestre,numérico,1827,0,0.0,sim,False,4,1,4,"[1, 4]",0
dim_data,mes,numérico,1827,0,0.0,sim,False,12,1,12,"[1, 12]",0
dim_data,ano_mes,categórico,1827,0,0.0,sim,False,60,2016-01,2020-12,livre,0
dim_data,dia_semana,numérico,1827,0,0.0,sim,False,7,1,7,"[1, 7]",0
dim_data,flag_fim_de_semana,booleano,1827,0,0.0,sim,False,2,false,true,true|false,0
dim_cliente,customer_id,texto,99441,0,0.0,sim,False,99441,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,livre,0
dim_cliente,customer_unique_id,texto,99441,0,0.0,sim,False,96096,0000366f3b9a7992bf8c76cfdf3221e2,ffffd2657e2aad2907e67c3e9daecbeb,livre,0


### Atributos que exigem atenção
Colunas obrigatórias com nulos ou com valores fora do domínio declarado.

In [ ]:
display(dq_gold[(dq_gold.fora_do_dominio > 0) | dq_gold.viola_obrigatoriedade]
        [["tabela", "atributo", "nulos", "minimo", "maximo", "dominio", "fora_do_dominio"]])

tabela,atributo,nulos,minimo,maximo,dominio,fora_do_dominio
dim_produto,peso_g,2,0,40425,"[1, 50000]",4
fato_pedido,dias_aprovacao,160,0.0,187.8825,"[0, 10]",48
fato_pedido,dias_postagem,1797,-171.219,125.7626,"[0, 30]",1544
fato_pedido,dias_transporte,2966,-16.0962,205.191,"[0, 60]",248
fato_pedido,dias_lead_time,2965,0.5334,209.6286,"[0, 90]",77
fato_pedido,dias_atraso,2965,-147,188,"[-60, 60]",113
fato_pedido,folga_dias,2965,-188,147,"[-60, 60]",113
fato_pagamento,parcelas,0,0,24,"[1, 24]",2
fato_pagamento,valor,0,0.00,13664.08,"[0.01, 15000]",9


## 3. Regras de qualidade
Cada regra é uma consulta SQL que conta violações. O resultado esperado indica se a violação é tolerada (tratada por uma regra de negócio) ou bloqueante.

In [ ]:
REGRAS = [
    ("unicidade", "Pedido duplicado na fato_pedido",
     f"SELECT count(*) - count(DISTINCT order_id) FROM {GOLD}.fato_pedido", "zero"),
    ("unicidade", "Item duplicado na fato_item_pedido",
     f"SELECT count(*) - count(DISTINCT order_id, order_item_id) FROM {GOLD}.fato_item_pedido", "zero"),
    ("unicidade", "Avaliações a mais na origem (tratado por RN10)",
     f"SELECT count(*) - count(DISTINCT order_id) FROM {BRONZE}.order_reviews", "tolerado"),
    ("unicidade", "review_id repetido em pedidos diferentes na origem",
     f"SELECT count(*) - count(DISTINCT review_id) FROM {BRONZE}.order_reviews", "tolerado"),
    ("integridade", "Item sem pedido correspondente",
     f"SELECT count(*) FROM {GOLD}.fato_item_pedido i LEFT ANTI JOIN {GOLD}.fato_pedido p USING (order_id)", "zero"),
    ("integridade", "Item com produto inexistente",
     f"SELECT count(*) FROM {GOLD}.fato_item_pedido i LEFT ANTI JOIN {GOLD}.dim_produto p USING (product_id)", "zero"),
    ("integridade", "Item com vendedor inexistente",
     f"SELECT count(*) FROM {GOLD}.fato_item_pedido i LEFT ANTI JOIN {GOLD}.dim_vendedor v USING (seller_id)", "zero"),
    ("integridade", "Pedido com cliente inexistente",
     f"SELECT count(*) FROM {GOLD}.fato_pedido p LEFT ANTI JOIN {GOLD}.dim_cliente c USING (customer_id)", "zero"),
    ("integridade", "Pagamento sem pedido correspondente",
     f"SELECT count(*) FROM {GOLD}.fato_pagamento g LEFT ANTI JOIN {GOLD}.fato_pedido p USING (order_id)", "zero"),
    ("integridade", "Avaliação de pedido inexistente",
     f"SELECT count(*) FROM {SILVER}.avaliacoes a LEFT ANTI JOIN {GOLD}.fato_pedido p USING (order_id)", "zero"),
    ("completude", "Pedido sem nenhum item (pedidos cancelados ou indisponíveis)",
     f"SELECT count(*) FROM {GOLD}.fato_pedido WHERE qtd_itens IS NULL", "tolerado"),
    ("completude", "Pedido entregue sem avaliação",
     f"SELECT count(*) FROM {GOLD}.fato_pedido WHERE flag_entregue AND nota_avaliacao IS NULL", "tolerado"),
    ("completude", "Pedido entregue de vendedor único sem distância (prefixo sem coordenada)",
     f"SELECT count(*) FROM {GOLD}.fato_pedido WHERE flag_entregue AND seller_id IS NOT NULL AND distancia_km IS NULL", "tolerado"),
    ("consistência", "Pedido entregue com etapa de duração negativa (RN07)",
     f"SELECT count(*) FROM {GOLD}.fato_pedido WHERE flag_entregue AND flag_inconsistencia_temporal", "tolerado"),
    ("consistência", "Status delivered sem data de entrega",
     f"SELECT count(*) FROM {GOLD}.fato_pedido WHERE status = 'delivered' AND data_entrega_key IS NULL", "tolerado"),
    ("consistência", "Data de entrega preenchida com status diferente de delivered",
     f"SELECT count(*) FROM {GOLD}.fato_pedido WHERE status <> 'delivered' AND data_entrega_key IS NOT NULL", "tolerado"),
    ("consistência", "Soma dos pagamentos difere do valor de itens mais frete em mais de R$ 1",
     f"""SELECT count(*) FROM (SELECT order_id, sum(valor) v FROM {GOLD}.fato_pagamento GROUP BY order_id) g
         JOIN {GOLD}.fato_pedido p USING (order_id) WHERE abs(g.v - (p.valor_itens + p.valor_frete)) > 1""", "tolerado"),
    ("acurácia", "Ponto de geolocalização fora do Brasil (tratado por RN12)",
     f"""SELECT count(*) FROM {BRONZE}.geolocation WHERE NOT (CAST(geolocation_lat AS DOUBLE) BETWEEN -33.75 AND 5.27
         AND CAST(geolocation_lng AS DOUBLE) BETWEEN -73.99 AND -34.79)""", "tolerado"),
    ("acurácia", "Prazo-limite de envio posterior a 2018",
     f"SELECT count(*) FROM {GOLD}.fato_item_pedido WHERE data_limite_envio_key > 20181231", "tolerado"),
    ("acurácia", "Produto com peso zero",
     f"SELECT count(*) FROM {GOLD}.dim_produto WHERE peso_g = 0", "tolerado"),
    ("atualidade", "Dias entre a última compra registrada e a data da análise",
     f"SELECT datediff(current_date(), max(to_date(CAST(data_compra_key AS STRING), 'yyyyMMdd'))) FROM {GOLD}.fato_pedido", "informativo"),
]
resultado = pd.DataFrame([{"dimensao": d, "regra": n, "violacoes": spark.sql(q).first()[0], "esperado": e, "consulta": q}
                          for d, n, q, e in REGRAS])
resultado["status"] = resultado.apply(
    lambda r: "FALHA" if r.esperado == "zero" and r.violacoes > 0 else ("ok" if r.esperado == "zero" else r.esperado), axis=1)
gravar(resultado, "dq_regras", "Qualidade: regras de unicidade, integridade, consistência, acurácia e atualidade")
display(resultado[["dimensao", "regra", "violacoes", "status"]])
assert (resultado.status != "FALHA").all(), "Regra bloqueante violada"

dimensao,regra,violacoes,status
unicidade,Pedido duplicado na fato_pedido,0,ok
unicidade,Item duplicado na fato_item_pedido,0,ok
unicidade,Avaliações a mais na origem (tratado por RN10),551,tolerado
unicidade,review_id repetido em pedidos diferentes na origem,814,tolerado
integridade,Item sem pedido correspondente,0,ok
integridade,Item com produto inexistente,0,ok
integridade,Item com vendedor inexistente,0,ok
integridade,Pedido com cliente inexistente,0,ok
integridade,Pagamento sem pedido correspondente,0,ok
integridade,Avaliação de pedido inexistente,0,ok
